# 手写 2SLS 估计程序

> 对应代码：`MathStatsCode/code_in_notes/Appendix.B/twosls_by_hand.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：编写程序实现两阶段最小二乘：先用工具变量预测内生变量，再对预测值做 OLS 并计算同方差标准误，最后与 ivregress 对比验证。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Appendix.B`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Appendix.B")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

**2SLS by hand**

本 notebook 用程序完整实现**两阶段最小二乘（2SLS）**，帮助理解 `ivregress` 背后的计算过程。程序 `twosls` 的要点：
- `syntax` 解析出被解释变量、内生变量（endo）、工具变量（instru）与控制变量（control）；
- **第一阶段**：对每个内生变量做工具变量回归 `reg x instru control` 并 `predict` 得到拟合值，把拟合值收集到宏中；
- **第二阶段**：用被解释变量对“内生变量的拟合值+控制变量”做 OLS，得到系数矩阵 `b`；
- **残差与方差**：用**原始**变量（而非拟合值）计算残差 `resid=y-Xb`，其样本方差作为 σ² 的估计；`matrix accum` 计算拟合值矩阵的交叉乘积 X̂'X̂（`matrix accum` 自动包含常数项）；
- 同方差假定下的方差公式 V=σ²(X̂'X̂)⁻¹（`invsym` 求对称逆），这也是 2SLS 的标准方差估计；
- 最后 `ereturn clear/post` 把系数与方差按标准 eclass 格式发布、`ereturn display` 输出结果表——这样自定义估计命令就能像官方命令一样与 `estat`、`esttab` 等工具兼容。


In [ ]:
%%stata
cap program drop twosls
program twosls, eclass
	version 15.0
	syntax varlist(max=1), endo(varlist) instru(varlist) /*
		*/control(varlist)
	tempname dep Xhat XhhX pred_endo pred_var b sigma2 V
	tempvar resid
	local `dep' "`varlist'"
	quietly{
		// get the predicted value of endo x
		local `pred_var' ""
		foreach x of varlist `endo'{
			reg `x' `instru' `control'
			predict `pred_endo'_`x'
			local `pred_var' "``pred_var'' `pred_endo'_`x'"
		}
		local `Xhat' "``pred_var'' `control'"
		// ols of y on Xhat, get b
		reg ``dep'' ``Xhat''
		matrix `b'=e(b)
		mat colnames `b'= `endo' `control' _cons
		///// compute variance under homoskedasticity
		// first compute residuals
		gen `resid'=``dep''
		foreach x of varlist `endo' `control'{
			replace `resid'=`resid'-`b'[1,colnumb(`b',"`x'")]*`x'
		}
		replace `resid'=`resid'-`b'[1,colnumb(`b',"_cons")]
		su `resid'
		scalar `sigma2'=r(Var)
		// then compute XhhX
		matrix accum `XhhX'=``Xhat''
		matrix `V'=`sigma2'*invsym(`XhhX')
		mat colnames `V'= `endo' `control' _cons
		mat rownames `V'= `endo' `control' _cons
		drop ``pred_var''
	}
	ereturn clear
	ereturn post `b' `V'
	ereturn local depvar "``dep''"
	ereturn display
end


**test the program**

准备测试环境：清空数据、关闭分页、`set obs 1000`。


In [ ]:
%%stata
clear
set more off
set obs 1000


**generate fake data**

生成模拟数据：`z~N(0,1)` 为工具变量，`u~N(0,1)` 是同时影响内生变量与结果的**共同冲击**（造成内生性）；`x1=z+u+rnormal()` 与工具变量相关且受 u 影响；`x2=z+0.5·rnormal()` 为控制变量；`y=2+x1+x2+u`。


In [ ]:
%%stata
gen z=rnormal()
gen u=rnormal()
gen x1=z+rnormal()+u
gen x2=z+0.5*rnormal()
gen y=2+x1+x2+u


**compare it with ivregress commmand**

对比两种估计：先跑官方命令 `ivregress 2sls y (x1=z) x2`（`(x1=z)` 表示用 z 作为 x1 的工具），再跑刚定义的 `twosls y, endo(x1) instru(z) control(x2)`——两者报告的点估计数值应当一致，说明自编程序正确复现了 2SLS。同时注意标准误是否一致：官方命令默认使用更稳健的小样本校正。


In [ ]:
%%stata
ivregress 2sls y (x1=z) x2
twosls y, endo(x1) instru(z) control(x2)
